In [9]:
import pandas as pd
import numpy as np
import sys
import os
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

sys.path.append(os.path.abspath(".."))

from src.modeling import (
    split_data,
    build_preprocessor,
    get_models,
    train_model,
    evaluate_model
)

df = pd.read_csv(
    "../data/insurance_data_clean.txt",
    low_memory=False
)

df["CapitalOutstanding"].dtype
df["CapitalOutstanding"].unique()[:10]

df["CapitalOutstanding"] = pd.to_numeric(
    df["CapitalOutstanding"],
    errors="coerce"
)

df["CapitalOutstanding"].isna().sum()

df["CapitalOutstanding"] = df["CapitalOutstanding"].fillna(
    df["CapitalOutstanding"].median()
)


df.head()


df["VehicleAge"] = 2026 - df["RegistrationYear"]

df = df[df["TotalClaims"] > 0]  # severity model requirement

MODEL 1: CLAIM SEVERITY (CORE TASK)

In [14]:
X_train, X_test, y_train, y_test = split_data(df, target="TotalClaims")

preprocessor = build_preprocessor(X_train)

models = get_models()

results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    print(X_train.isna().sum().sum())

    pipeline.fit(X_train, y_train)

    metrics = evaluate_model(pipeline, X_test, y_test)

    results[name] = metrics

results

2230


ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

Model Comparison Table

In [ ]:
pd.DataFrame(results).T

BEST MODEL SELECTION

In [ ]:
best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models["XGBoost"])
])

best_model.fit(X_train, y_train)

FEATURE IMPORTANCE

SHAP Analysis

In [ ]:
explainer = shap.Explainer(best_model.named_steps["model"])

X_transformed = preprocessor.transform(X_test)

shap_values = explainer(X_transformed)

SHAP Plot

In [ ]:
shap.summary_plot(shap_values, X_transformed)

RISK-BASED PRICING MODEL

In [ ]:
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)
X_train, X_test, y_train, y_test = split_data(df, target="HasClaim")

from sklearn.ensemble import RandomForestClassifier

clf = Pipeline([
    ("preprocessor", build_preprocessor(X_train)),
    ("model", RandomForestClassifier())
])



clf.fit(X_train, y_train)

claim_prob = clf.predict_proba(X_test)[:, 1]

predicted_severity = best_model.predict(X_test)

expense_loading = 500
profit_margin = 300

premium = (
    claim_prob * predicted_severity
) + expense_loading + profit_margin



BUSINESS INSIGHT SECTION

Key Insight

Vehicles with higher age and higher engine capacity significantly increase predicted claim severity, suggesting targeted premium adjustment strategies.

Pricing Recommendation

The risk-based pricing model shows that combining probability of claim with severity provides more accurate premiums than flat pricing.